In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
default_path = os.path.join(path, 'Q3_data.csv')
df_default = pd.read_csv(default_path)

In [ ]:
# Task 2: Write your code here:
df_default.head()

In [ ]:
# Task 3: Write your code here:
df_default.info()

In [ ]:
# Task 4: Write your code here:
df_default.describe()

In [ ]:
# Task 1: Write your code here:
#Check for missing values
print("Missing values:")
df_default.isnull().sum()

In [ ]:
#df_clean = df_default.copy()
categ_cols = df_default.select_dtypes(include='object').columns
num_cols = df_default.select_dtypes(include='number').columns


df_default[num_cols] = df_default[num_cols].fillna(df_default[num_cols].mean())
df_default[categ_cols] = df_default[categ_cols].fillna(df_default[categ_cols].mode())

df_clean = df_default.copy()
print(f"Shape after cleaning: {df_clean.shape}")


In [ ]:
print("Missing values:")
df_clean.isnull().sum()

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

categ_cols = df_default.select_dtypes(include='object').columns


label_encoders = {}
for col in categ_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le
df_clean

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 5: Write your code here:
df_clean.value_counts('Target')

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Target", axis=1).astype(float)
y = df_clean["Target"].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
!pip install catboost
from catboost import CatBoostClassifier
from sklearn.metrics import cross_val_score


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/5")
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Random Forest Regressor
  cbc_model = CatBoostClassifier(
    n_estimators=100
    )
  cbc_model.fit(X_train, y_train)

  # Predict
  y_pred = cbc_model.predict(X_test)

  # Calculate metrics
  cross=cross_val_score(y_test, y_pred)

  print(f"\n{cbc_model}:")
  print(f"  MAE:  {np.mean(cbc_model['cross']):.4f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': cbc_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: